In [1]:
import pandas  as pd 
import numpy as np

data = pd.read_csv("data/processed/development_processed.csv")

In [3]:
data.shape

(79997, 12)

In [5]:
import platform
import psutil

print("=== SYSTEM INFO ===")
print("OS:", platform.system(), platform.release())
print("Machine:", platform.machine())
print("Processor:", platform.processor())

# CPU
print("\n=== CPU ===")
print("Physical cores:", psutil.cpu_count(logical=False))
print("Logical cores:", psutil.cpu_count(logical=True))
print("Max frequency (MHz):", psutil.cpu_freq().max)

# RAM
mem = psutil.virtual_memory()
print("\n=== RAM ===")
print(f"Total: {mem.total / (1024**3):.2f} GB")
print(f"Available: {mem.available / (1024**3):.2f} GB")

# GPU (se presente, via PyTorch)
print("\n=== GPU ===")
try:
	import torch
	if torch.cuda.is_available():
		print("CUDA available:", torch.cuda.is_available())
		print("GPU count:", torch.cuda.device_count())
		for i in range(torch.cuda.device_count()):
			print(f"GPU {i}:", torch.cuda.get_device_name(i))
	else:
		print("No CUDA GPU detected.")
except Exception as e:
	print("PyTorch not available or GPU not detected.")


=== SYSTEM INFO ===
OS: Windows 10
Machine: AMD64
Processor: AMD64 Family 23 Model 104 Stepping 1, AuthenticAMD

=== CPU ===
Physical cores: 8
Logical cores: 16
Max frequency (MHz): 1801.0

=== RAM ===
Total: 31.33 GB
Available: 8.09 GB

=== GPU ===
No CUDA GPU detected.


In [1]:
import pandas as pd

roberta = pd.read_csv("data/submission/submission_roberta_full_preprocesed.csv")
deberta_raw = pd.read_csv("data/submission/submission_daBerta_raw.csv")
deberta_proc = pd.read_csv("deberta_processed/submission_deberta_processed.csv")

# sicurezza: merge su Id
df = (
    roberta
    .merge(deberta_raw, on="Id", suffixes=("_roberta", "_deberta_raw"))
    .merge(deberta_proc, on="Id")
    .rename(columns={"Predicted": "Predicted_deberta_proc"})
)

print(df.head())
print("Tot samples:", len(df))


   Id  Predicted_roberta  Predicted_deberta_raw  Predicted_deberta_proc
0   0                  1                      3                       5
1   1                  2                      2                       2
2   2                  0                      0                       0
3   3                  1                      0                       1
4   4                  0                      0                       0
Tot samples: 20000


In [2]:
df["rob_vs_raw"] = df["Predicted_roberta"] == df["Predicted_deberta_raw"]
df["rob_vs_proc"] = df["Predicted_roberta"] == df["Predicted_deberta_proc"]
df["raw_vs_proc"] = df["Predicted_deberta_raw"] == df["Predicted_deberta_proc"]

print("Roberta vs DeBERTa raw  - same:", df["rob_vs_raw"].sum())
print("Roberta vs DeBERTa proc - same:", df["rob_vs_proc"].sum())
print("Raw vs Proc            - same:", df["raw_vs_proc"].sum())


Roberta vs DeBERTa raw  - same: 17302
Roberta vs DeBERTa proc - same: 17158
Raw vs Proc            - same: 17592


In [3]:
n = len(df)

print("Agreement Roberta vs DeBERTa raw :",
      df["rob_vs_raw"].mean().round(4))

print("Agreement Roberta vs DeBERTa proc:",
      df["rob_vs_proc"].mean().round(4))

print("Agreement Raw vs Proc:",
      df["raw_vs_proc"].mean().round(4))


Agreement Roberta vs DeBERTa raw : 0.8651
Agreement Roberta vs DeBERTa proc: 0.8579
Agreement Raw vs Proc: 0.8796


In [4]:
df_diff = df[
    (df["Predicted_roberta"] != df["Predicted_deberta_raw"]) |
    (df["Predicted_roberta"] != df["Predicted_deberta_proc"])
]

print("Samples with disagreement:", len(df_diff))
df_diff.head()


Samples with disagreement: 3825


,Id,Predicted_roberta,Predicted_deberta_raw,Predicted_deberta_proc,rob_vs_raw,rob_vs_proc,raw_vs_proc
0,0,1,3,5,False,False,False
3,3,1,0,1,False,True,False
6,6,1,1,3,True,False,False
9,9,5,0,5,False,True,False
13,13,2,1,2,False,True,False


In [5]:
df_diff["case"] = (
    (df_diff["Predicted_roberta"] != df_diff["Predicted_deberta_raw"]).astype(int) +
    (df_diff["Predicted_roberta"] != df_diff["Predicted_deberta_proc"]).astype(int)
)

df_diff["case"].value_counts()


C:\Users\msist\AppData\Local\Temp\ipykernel_15104\3591539248.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_diff["case"] = (


case
1    2110
2    1715
Name: count, dtype: int64

In [1]:
import pandas as pd
import hashlib

def text_hash(title, article):
	text = (str(title) + " " + str(article)).lower().strip()
	return hashlib.md5(text.encode("utf-8")).hexdigest()

# carica DEV
df_dev = pd.read_csv("data/raw/development.csv")

df_dev["text_hash"] = df_dev.apply(
	lambda r: text_hash(r["title"], r["article"]), axis=1
)

# gruppi duplicati
dup_groups = df_dev.groupby("text_hash")

# tieni solo quelli con più di una occorrenza
dups = dup_groups.filter(lambda g: len(g) > 1)

print("Duplicated DEV rows:", len(dups))
print("Unique duplicated texts:", dups["text_hash"].nunique())


Duplicated DEV rows: 5522
Unique duplicated texts: 2554


In [2]:
label_conflicts = (
	dups.groupby("text_hash")["label"]
	.nunique()
	.reset_index(name="n_labels")
)

conflicting = label_conflicts[label_conflicts["n_labels"] > 1]

print("Conflicting duplicated texts:", len(conflicting))


Conflicting duplicated texts: 1517


In [3]:
conflict_ids = set(conflicting["text_hash"])

conflict_rows = dups[dups["text_hash"].isin(conflict_ids)]

print(conflict_rows["label"].value_counts(normalize=True))


label
5    0.340646
0    0.216941
3    0.200488
1    0.089580
2    0.065814
6    0.047227
4    0.039305
Name: proportion, dtype: float64


In [4]:
df_eval = pd.read_csv("data/raw/evaluation.csv")

df_eval["text_hash"] = df_eval.apply(
	lambda r: text_hash(r["title"], r["article"]), axis=1
)

# overlap DEV ↔ EVAL
overlap = df_eval["text_hash"].isin(df_dev["text_hash"])

print("EVAL rows identical to DEV:", overlap.sum())
print("Percentage:", overlap.mean())


EVAL rows identical to DEV: 1291
Percentage: 0.06455


In [5]:
eval_conflict_overlap = df_eval["text_hash"].isin(conflicting["text_hash"])
print("EVAL rows matching conflicting DEV texts:", eval_conflict_overlap.sum())


EVAL rows matching conflicting DEV texts: 109


In [6]:
import pandas as pd
import hashlib

def text_hash(title, article):
	text = (str(title) + " " + str(article)).lower().strip()
	return hashlib.md5(text.encode("utf-8")).hexdigest()

# carica EVAL
df_eval = pd.read_csv("data/raw/evaluation.csv")

df_eval["text_hash"] = df_eval.apply(
	lambda r: text_hash(r["title"], r["article"]), axis=1
)

# gruppi duplicati
eval_groups = df_eval.groupby("text_hash")

# solo testi che compaiono più volte
eval_dups = eval_groups.filter(lambda g: len(g) > 1)

print("Duplicated EVAL rows:", len(eval_dups))
print("Unique duplicated texts (EVAL):", eval_dups["text_hash"].nunique())
print("Percentage duplicated rows:", len(eval_dups) / len(df_eval))


Duplicated EVAL rows: 346
Unique duplicated texts (EVAL): 159
Percentage duplicated rows: 0.0173


In [7]:
cluster_sizes = (
	eval_dups.groupby("text_hash")
	.size()
	.value_counts()
	.sort_index()
)

print(cluster_sizes)


2    147
3      7
4      2
7      1
8      2
Name: count, dtype: int64


In [8]:
eval_dup_conflict = eval_dups["text_hash"].isin(conflicting["text_hash"])
print(
	"EVAL duplicated rows matching conflicting DEV texts:",
	eval_dup_conflict.sum()
)


EVAL duplicated rows matching conflicting DEV texts: 37
